[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabin2004/Machine-Learning-Bootcamp/blob/main/Module_06_Model_Evaluation/01_model_evaluation.ipynb)

# Episode 16 & 17 – Model Evaluation & Validation

**Machine Learning Bootcamp** | Module 06

---

## 🎯 Learning Objectives
- Apply k-fold and stratified cross-validation
- Compute and interpret classification and regression metrics
- Plot ROC curves and learning curves
- Diagnose overfitting vs. underfitting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, learning_curve
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

sns.set_theme(style='whitegrid')

## 1. Cross-Validation

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_s = StandardScaler().fit_transform(X)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in [('LogisticRegression', LogisticRegression(max_iter=1000)),
                  ('RandomForest',       RandomForestClassifier(n_estimators=100, random_state=42))]:
    scores = cross_val_score(clf, X_s, y, cv=skf, scoring='f1')
    print(f'{name:<25}  F1 per fold: {scores.round(4)}  |  Mean={scores.mean():.4f} ± {scores.std():.4f}')

## 2. Classification Metrics Deep-Dive

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_s, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print('Accuracy: ', accuracy_score(y_test, y_pred).round(4))
print('Precision:', precision_score(y_test, y_pred).round(4))
print('Recall:   ', recall_score(y_test, y_pred).round(4))
print('F1 Score: ', f1_score(y_test, y_pred).round(4))
print()
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.tight_layout(); plt.show()

## 3. ROC Curve & AUC

In [ ]:
y_proba_rf = rf.predict_proba(X_test)[:, 1]

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

plt.figure(figsize=(7, 6))
for name, proba in [('Random Forest', y_proba_rf), ('Logistic Regression', y_proba_lr)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

## 4. Learning Curves

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(n_estimators=50, random_state=42),
    X_s, y, cv=5, scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 10)
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='steelblue', label='Train F1')
plt.fill_between(train_sizes,
                 train_scores.mean(1) - train_scores.std(1),
                 train_scores.mean(1) + train_scores.std(1), alpha=0.2, color='steelblue')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='tomato',    label='Val F1')
plt.fill_between(train_sizes,
                 val_scores.mean(1) - val_scores.std(1),
                 val_scores.mean(1) + val_scores.std(1), alpha=0.2, color='tomato')
plt.xlabel('Training Size'); plt.ylabel('F1 Score')
plt.title('Learning Curve – Random Forest')
plt.legend(); plt.tight_layout(); plt.show()

## 🏋️ Exercises

1. Plot a **Precision-Recall curve** and compare it to the ROC curve. Which is more informative for imbalanced classes?
2. Change the decision threshold of the Random Forest to 0.3. How do precision and recall change?
3. Create a learning curve for Logistic Regression. Does it suffer from high bias or high variance?

---
**Next ▶ [Module 07 – Unsupervised Learning](../Module_07_Unsupervised_Learning/01_kmeans.ipynb)**